# Build a Glue ETL job that cleans, deduplicates, and converts CSV to Parquet

The payments team handed off raw CSVs from three processors this morning. Same schema, but each processor stamps records in a different way: `processor_alpha` writes ISO 8601 timestamps, `processor_beta` writes epoch seconds, and `processor_gamma` writes `MM/DD/YYYY`. The duplicates from reissue events overlap across files, and guest-checkout rows have a null `customer_id`. The analytics team queries this dataset at 9 AM tomorrow.

Your job is the Glue ETL job that turns those three messy CSVs into one clean Parquet dataset partitioned by year and month. Glue 4.0 PySpark, one job run per attempt, no Glue crawler, no Athena. Lab 05 picks up Athena.

The four deliverables map to four checkpoints:
1. Three raw CSVs seeded under `raw/` with the imperfections the script must address: 10+ duplicate `order_id` values across files, null `customer_id` rows, and three different date formats.
2. A Glue ETL job created with `GlueVersion=4.0`, the right IAM role, `WorkerType=G.1X`, and a `Command.ScriptLocation` pointing at the uploaded PySpark script.
3. A job run that finishes with `JobRunState=SUCCEEDED` and a non-zero `ExecutionTime`.
4. Curated Parquet under `curated/year=YYYY/month=MM/` with the deduplicated row count, ISO 8601 timestamps, and the null sentinel in place.

**Time.** About 60 minutes hands-on. Most of the wall-clock time is the Glue 10-minute minimum job run, not your typing. Budget up to 90 minutes if you hit a script bug and need a second run.

**Cost.** This lab costs about 15 to 50 cents. Glue ETL bills at $0.44 per DPU-hour with a 10-minute minimum, so a 2-DPU job costs about 15 cents per run. Plan for one to three runs while you debug the script. The cleanup cell deletes the job so the bill stops the instant you finish.

This lab maps to AWS DEA-C01 Domain 1: Data Ingestion and Transformation (34% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import csv
import getpass
import io
import json
import random
import re
import time
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "glue-etl-transforms"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[3].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
ROLE_NAME = f"labrun-{LAB_ID}-job-role"
INLINE_POLICY_NAME = f"labrun-{LAB_ID}-job-inline"
DATABASE_NAME = f"labrun_{LAB_ID.replace('-', '_')}_db"
JOB_NAME = f"labrun-{LAB_ID}-job"
CURATED_TABLE_NAME = "orders_curated"
BUCKET_NAME = None  # resolved after STS once the account ID is known

# Seed config. Deterministic so Checkpoint 4 can compute the expected dedup
# count from the raw CSVs without coupling to Checkpoint 1.
SEED = 42
ROWS_PER_FILE = 120
DUPLICATES_FROM_ALPHA_IN_GAMMA = 15
NULLS_PER_FILE = 4


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All DEA-C01 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first IAM call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID. S3 bucket names
# must be globally unique.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The manifest is module-level and in reverse-creation order. Lab 04 has no
# critical (hourly-billed) resources. Glue ETL jobs bill only while running
# and the lab caps each run at the 10-minute Glue minimum.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).
#
# Note: labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table. It does NOT yet support glue_job. A
# LabAdapter subclass extending AwsCleanupAdapter is defined in the cleanup
# cell at the bottom of this notebook and passed to run_cleanup. The new
# resource type is still declared declaratively here so the canonical
# summary, atexit handler, and orphan scan all see it.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="glue_table",
        id=CURATED_TABLE_NAME,
        region=REGION,
        parent=DATABASE_NAME,
        cli_delete_command=(
            f"aws glue delete-table --database-name {DATABASE_NAME} "
            f"--name {CURATED_TABLE_NAME}"
        ),
    ),
    CleanupResource(
        type="glue_job",
        id=JOB_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {JOB_NAME}",
    ),
    CleanupResource(
        type="glue_database",
        id=DATABASE_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DATABASE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is the
    safety net for kernel crashes. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources or")
    print("collide on the role and job names. Run the cleanup cell at the")
    print("bottom of this notebook first, or remove them manually with the")
    print("AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Seed the bucket with three CSVs that have the imperfections the script must fix

The payments team will send real files in production, but for this lab you write the seed yourself so the failure modes are deterministic. The PySpark script you ship at the end of the lab has to handle three specific imperfections:

1. **Duplicate `order_id` values across files.** Reissue events tag the same `order_id` again in a later file. Around 15 order IDs appear in both `processor_alpha.csv` and `processor_gamma.csv`. The script must drop the duplicates on `order_id` and keep one row per ID.
2. **Null `customer_id` rows.** Guest-checkout transactions land with no `customer_id`. Each file has roughly 4 such rows. The script must fill those with the sentinel string `UNKNOWN` so downstream Parquet readers do not trip on null partition keys later.
3. **Three different date formats in `created_at`.** `processor_alpha` writes ISO 8601 (`2024-11-01T00:00:00Z`), `processor_beta` writes epoch seconds (`1730419200`), `processor_gamma` writes `MM/DD/YYYY` (`11/01/2024`). The script normalizes all three into ISO 8601 strings in the output, using `coalesce` over three `to_timestamp` calls.

Columns in every file: `order_id`, `customer_id`, `amount_cents`, `currency`, `processor_name`, `created_at`.

The deterministic seed (`random.seed(42)`) means every student runs the exact same data, so the row-count math in Checkpoint 4 lines up cleanly. After seeding, Checkpoint 1 walks the three files and asserts the imperfections are actually there.


In [ ]:
# NBVAL_SKIP
# Task 1: Create the bucket and upload three deterministically-generated CSVs
# under raw/ with the imperfections the PySpark script must fix.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the bucket. us-east-1 rejects LocationConstraint; other regions
# require it. All DEA-C01 labs run in us-east-1, so the simple call works.
# YOUR CODE: Create the S3 bucket with s3.create_bucket(Bucket=BUCKET_NAME).

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# Deterministic synthetic data. random.seed makes the row counts predictable
# so Checkpoint 4's dedup math holds.
random.seed(SEED)
COLUMNS = ["order_id", "customer_id", "amount_cents", "currency", "processor_name", "created_at"]
CURRENCIES = ["USD", "USD", "USD", "EUR", "GBP"]
BASE_EPOCH = 1730419200  # 2024-11-01T00:00:00Z
STEP_SECONDS = 86400  # 1 day per row so dates span Nov 2024 through Feb/Mar 2025


def _amount():
    return random.randint(100, 99999)


def _customer(row_index, null_rows):
    if row_index in null_rows:
        return ""
    return f"cust_{random.randint(1000, 9999)}"


def _build_rows(processor_name, date_fn, start_offset_days, count, null_rows, order_id_overrides=None):
    rows = []
    for i in range(count):
        order_id = (
            order_id_overrides[i]
            if order_id_overrides and i in order_id_overrides
            else f"ord_{processor_name[:5]}_{i:04d}"
        )
        rows.append([
            order_id,
            _customer(i, null_rows),
            _amount(),
            random.choice(CURRENCIES),
            processor_name,
            date_fn(start_offset_days + i),
        ])
    return rows


def _iso_date(offset_days):
    t = BASE_EPOCH + offset_days * STEP_SECONDS
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(t))


def _epoch_date(offset_days):
    t = BASE_EPOCH + offset_days * STEP_SECONDS
    return str(t)


def _slashed_date(offset_days):
    t = BASE_EPOCH + offset_days * STEP_SECONDS
    return time.strftime("%m/%d/%Y", time.gmtime(t))


# Pick deterministic indexes for nulls in each file.
alpha_nulls = set(random.sample(range(ROWS_PER_FILE), NULLS_PER_FILE))
beta_nulls = set(random.sample(range(ROWS_PER_FILE), NULLS_PER_FILE))
gamma_nulls = set(random.sample(range(ROWS_PER_FILE), NULLS_PER_FILE))

alpha_rows = _build_rows("processor_alpha", _iso_date, 0, ROWS_PER_FILE, alpha_nulls)
beta_rows = _build_rows("processor_beta", _epoch_date, 30, ROWS_PER_FILE, beta_nulls)

# Gamma reuses 15 of alpha's order_ids so dedup has something to remove.
gamma_overrides = {
    i: alpha_rows[i][0] for i in range(DUPLICATES_FROM_ALPHA_IN_GAMMA)
}
gamma_rows = _build_rows(
    "processor_gamma", _slashed_date, 14, ROWS_PER_FILE, gamma_nulls,
    order_id_overrides=gamma_overrides,
)


def _csv_bytes(rows):
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(COLUMNS)
    for row in rows:
        writer.writerow(row)
    return buf.getvalue().encode("utf-8")


ALPHA_KEY = "raw/processor_alpha.csv"
BETA_KEY = "raw/processor_beta.csv"
GAMMA_KEY = "raw/processor_gamma.csv"

alpha_csv = _csv_bytes(alpha_rows)
beta_csv = _csv_bytes(beta_rows)
gamma_csv = _csv_bytes(gamma_rows)

# YOUR CODE: Upload the three CSV byte payloads to s3://{BUCKET_NAME}/raw/.
# Use s3.put_object(Bucket=BUCKET_NAME, Key=ALPHA_KEY, Body=alpha_csv) and the
# matching calls for beta and gamma. Encoding is already utf-8 bytes.

print(f"Seeded 3 files under s3://{BUCKET_NAME}/raw/")
print(f"  - {ALPHA_KEY} ({ROWS_PER_FILE} rows, ISO 8601 dates)")
print(f"  - {BETA_KEY} ({ROWS_PER_FILE} rows, epoch seconds dates)")
print(f"  - {GAMMA_KEY} ({ROWS_PER_FILE} rows, MM/DD/YYYY dates, {DUPLICATES_FROM_ALPHA_IN_GAMMA} order_ids shared with alpha)")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: three CSVs exist under raw/, totaling >= 300 rows, with
# >= 10 duplicate order_ids across files, at least one null customer_id per
# file, and three different date formats sniffed from row 1 of each file.

ISO_DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")
EPOCH_DATE_RE = re.compile(r"^\d{9,10}$")
SLASHED_DATE_RE = re.compile(r"^\d{1,2}/\d{1,2}/\d{4}$")


def checkpoint_1(session):
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        expected_keys = ["raw/processor_alpha.csv", "raw/processor_beta.csv", "raw/processor_gamma.csv"]
        contents_by_key: dict[str, str] = {}
        for key in expected_keys:
            try:
                obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
            except ClientError as e:
                code = e.response["Error"]["Code"]
                if code in ("NoSuchKey", "NoSuchBucket", "404"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=f"Object {key} is missing from bucket {BUCKET_NAME}.",
                    )
                return CheckpointResult(status="error", error_reason=str(e))
            contents_by_key[key] = obj["Body"].read().decode("utf-8")

        # Parse each CSV, count rows, count nulls per file, and collect order_ids.
        total_rows = 0
        per_file_order_ids: dict[str, list[str]] = {}
        per_file_nulls: dict[str, int] = {}
        per_file_first_date: dict[str, str] = {}
        for key, body in contents_by_key.items():
            reader = csv.DictReader(io.StringIO(body))
            rows = list(reader)
            per_file_order_ids[key] = [r["order_id"] for r in rows]
            per_file_nulls[key] = sum(
                1 for r in rows if not r.get("customer_id") or r["customer_id"].strip() == ""
            )
            per_file_first_date[key] = rows[0]["created_at"] if rows else ""
            total_rows += len(rows)

        if total_rows < 300:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Total rows across the three files is {total_rows}, expected >= 300. "
                    f"Re-run the seed cell."
                ),
            )

        # Cross-file duplicates: order_ids that appear in more than one file.
        order_id_to_files: dict[str, set[str]] = {}
        for key, ids in per_file_order_ids.items():
            for oid in ids:
                order_id_to_files.setdefault(oid, set()).add(key)
        cross_file_duplicates = sum(1 for files in order_id_to_files.values() if len(files) > 1)
        if cross_file_duplicates < 10:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Found {cross_file_duplicates} order_ids that appear in more than one file. "
                    f"The seed expects at least 10 cross-file duplicates so the PySpark script "
                    f"has something to dedup. Re-run the seed cell with the unchanged constants."
                ),
            )

        # Per-file null customer_id rows.
        for key, count in per_file_nulls.items():
            if count < 1:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{key} has 0 rows with null customer_id. The seed expects at least one "
                        f"per file so the PySpark script has nulls to fill with the sentinel."
                    ),
                )

        # Date-format sniff. row-1 of each file should match a different regex.
        alpha_first = per_file_first_date["raw/processor_alpha.csv"]
        beta_first = per_file_first_date["raw/processor_beta.csv"]
        gamma_first = per_file_first_date["raw/processor_gamma.csv"]
        if not ISO_DATE_RE.match(alpha_first):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"processor_alpha row 1 created_at is {alpha_first!r}, expected ISO 8601 "
                    f"like 2024-11-01T00:00:00Z. The seed cell defines _iso_date for alpha."
                ),
            )
        if not EPOCH_DATE_RE.match(beta_first):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"processor_beta row 1 created_at is {beta_first!r}, expected epoch seconds "
                    f"like 1730419200. The seed cell defines _epoch_date for beta."
                ),
            )
        if not SLASHED_DATE_RE.match(gamma_first):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"processor_gamma row 1 created_at is {gamma_first!r}, expected MM/DD/YYYY "
                    f"like 11/01/2024. The seed cell defines _slashed_date for gamma."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint runs five independent checks on the raw bucket: three files exist, total rows are at least 300, at least 10 order_ids appear in more than one file, every file has at least one null customer_id, and the three first-row date formats are distinct. Read the failure reason to see which check fired. The most common miss on first run is the bucket itself not being created (or being created without the tag), which surfaces as a `NoSuchKey` on the first `s3.get_object` call.

</details>

<details><summary>Hint 2 (direction)</summary>

The seed cell already builds the imperfect data in memory. Your two pieces of work in the cell are: call `s3.create_bucket(Bucket=BUCKET_NAME)` (no `LocationConstraint` in us-east-1) and call `s3.put_object` three times to upload `alpha_csv`, `beta_csv`, and `gamma_csv` to the right keys. The keys are already defined as `ALPHA_KEY`, `BETA_KEY`, `GAMMA_KEY`. The CSV byte payloads are already utf-8 encoded.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

s3.create_bucket(Bucket=BUCKET_NAME)
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

s3.put_object(Bucket=BUCKET_NAME, Key=ALPHA_KEY, Body=alpha_csv)
s3.put_object(Bucket=BUCKET_NAME, Key=BETA_KEY, Body=beta_csv)
s3.put_object(Bucket=BUCKET_NAME, Key=GAMMA_KEY, Body=gamma_csv)
```

If `create_bucket` returns `BucketAlreadyOwnedByYou`, the orphan scan in setup should have caught a leftover from a prior run; that path is treated as an error here on purpose. If you hit it anyway, run the cleanup cell at the bottom of the notebook to clear state, then re-run setup and Task 1.

</details>


**Wallet check.** Still at $0.00. The bucket itself is free; you only pay for storage and request volume. Three CSV objects of ~10 KB each fit inside the always-free S3 tier by a factor of millions. The morning coffee continues to outpace this lab.


## Task 2: Build the IAM role and create the Glue ETL job

Three pieces of plumbing to land in this task:

1. **IAM role** named `labrun-glue-etl-transforms-job-role` with the Glue trust policy (`Principal.Service` is `glue.amazonaws.com`, `Action` is `sts:AssumeRole`). Attach the managed policy `AWSGlueServiceRole` and one inline policy `labrun-glue-etl-transforms-job-inline` granting the S3 actions Glue needs (`s3:GetObject`, `s3:PutObject`, `s3:DeleteObject`, `s3:ListBucket`) on the lab bucket and its `/*` ARN.
2. **Glue Data Catalog database** named `labrun_glue_etl_transforms_db`. The PySpark job writes to S3 directly; the database holds the curated table that the script registers manually after the job finishes (Task 4). Glue databases are free and tag-trackable.
3. **The PySpark transform script** uploaded to `s3://{bucket}/scripts/transform.py`. The script body is in the next cell as `PYSPARK_SCRIPT`. Two reasons the script lives in S3: Glue requires `Command.ScriptLocation` to point at a real object at job-create time, and a redeployed script on a re-run is one `s3.put_object` away.
4. **The Glue ETL job** itself, created with `glue.create_job`. Required parameters: `Name`, `Role` (the job role ARN), `Command={"Name": "glueetl", "ScriptLocation": "s3://.../scripts/transform.py", "PythonVersion": "3"}`, `GlueVersion="4.0"`, `WorkerType="G.1X"`, `NumberOfWorkers=2`, and the lab tag. `DefaultArguments` carries `--BUCKET_NAME` so the script knows which bucket to read and write.

After creating the role, sleep 15 seconds before the next cell. IAM role propagation is asynchronous and Glue validates the role at job-create time. Skipping the sleep is the most common false-FAIL on Checkpoint 2.


In [ ]:
# NBVAL_SKIP
# Task 2: Upload the PySpark script, create the IAM role + Glue database,
# and create the Glue ETL job.

PYSPARK_SCRIPT = '''
import sys
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import (
    coalesce,
    col,
    date_format,
    from_unixtime,
    lit,
    month,
    to_timestamp,
    when,
    year,
)

args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET_NAME"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session
job = Job(glue_ctx)
job.init(args["JOB_NAME"], args)

bucket = args["BUCKET_NAME"]
raw_path = "s3://" + bucket + "/raw/"
curated_path = "s3://" + bucket + "/curated/"

df = spark.read.option("header", "true").csv(raw_path)

# Normalize the mixed-format date column using coalesce of three to_timestamp
# calls: ISO 8601, epoch seconds (cast to long then from_unixtime), MM/DD/YYYY.
df = df.withColumn(
    "created_at_ts",
    coalesce(
        to_timestamp(col("created_at"), "yyyy-MM-dd'T'HH:mm:ss'Z'"),
        to_timestamp(from_unixtime(col("created_at").cast("long"))),
        to_timestamp(col("created_at"), "MM/dd/yyyy"),
    ),
)

# Reformat created_at as ISO 8601 string in the output.
df = df.withColumn(
    "created_at",
    date_format(col("created_at_ts"), "yyyy-MM-dd'T'HH:mm:ss'Z'"),
)

# Fill null customer_id with the sentinel.
df = df.withColumn(
    "customer_id",
    when(
        col("customer_id").isNull() | (col("customer_id") == ""),
        lit("UNKNOWN"),
    ).otherwise(col("customer_id")),
)

# Partition columns derived from the parsed timestamp.
df = df.withColumn("year", year(col("created_at_ts")))
df = df.withColumn("month", month(col("created_at_ts")))

# Dedup on order_id. dropDuplicates keeps the first occurrence per partition,
# which is good enough for this lab.
df_dedup = df.dropDuplicates(["order_id"])

# Drop the helper timestamp column before write.
df_out = df_dedup.drop("created_at_ts")

df_out.write.mode("overwrite").partitionBy("year", "month").parquet(curated_path)

job.commit()
'''.lstrip()

SCRIPT_KEY = "scripts/transform.py"
script_uri = f"s3://{BUCKET_NAME}/{SCRIPT_KEY}"

# YOUR CODE: Upload the PySpark script to s3://{BUCKET_NAME}/scripts/transform.py
# with s3.put_object(Bucket=BUCKET_NAME, Key=SCRIPT_KEY, Body=PYSPARK_SCRIPT.encode("utf-8")).

print(f"Script uploaded: {script_uri}")

# Build the Glue trust policy. Principal is the Glue service.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "glue.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the IAM role with iam.create_role(
#   RoleName=ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).

# Attach the AWS-managed AWSGlueServiceRole policy. Glue ETL jobs need it
# for CloudWatch Logs + Glue catalog access on top of your inline S3 grants.
iam.attach_role_policy(
    RoleName=ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole",
)

# Inline policy: S3 actions on the lab bucket and its /* ARN.
bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:PutObject",
                "s3:DeleteObject",
                "s3:ListBucket",
            ],
            "Resource": [bucket_arn, f"{bucket_arn}/*"],
        }
    ],
}

# YOUR CODE: Attach the inline policy with iam.put_role_policy(
#   RoleName=ROLE_NAME,
#   PolicyName=INLINE_POLICY_NAME,
#   PolicyDocument=json.dumps(inline_policy),
# ).

print(f"Role created: {ROLE_NAME}")
print("Managed policy AWSGlueServiceRole attached.")
print("Inline S3 policy attached.")

# Glue database for the curated table registered in Task 4.
try:
    glue.create_database(
        DatabaseInput={
            "Name": DATABASE_NAME,
            "Description": f"Curated table catalog for {LAB_ID}",
        },
    )
    glue.tag_resource(
        ResourceArn=f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DATABASE_NAME}",
        TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Glue database created and tagged: {DATABASE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DATABASE_NAME} already exists, reusing.")
    else:
        raise

# IAM role propagation. Glue validates the role at job-create time.
# Skipping this sleep is the most common false-FAIL on Checkpoint 2.
print("Your IAM role is stuck in traffic, give it 15 seconds...")
time.sleep(15)
print("Role is ready.")

role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"

# YOUR CODE: Create the Glue ETL job with glue.create_job(
#   Name=JOB_NAME,
#   Role=role_arn,
#   Command={"Name": "glueetl", "ScriptLocation": script_uri, "PythonVersion": "3"},
#   DefaultArguments={"--BUCKET_NAME": BUCKET_NAME, "--enable-metrics": "true"},
#   GlueVersion="4.0",
#   WorkerType="G.1X",
#   NumberOfWorkers=2,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

print(f"Glue ETL job created: {JOB_NAME}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Glue ETL job exists with the right Role ARN, glueetl command,
# script location, GlueVersion=4.0, WorkerType=G.1X, and 2 to 4 workers.

REQUIRED_NUM_WORKERS_RANGE = (2, 4)


def checkpoint_2(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job_resp = glue_client.get_job(JobName=JOB_NAME)
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Glue job {JOB_NAME} does not exist. Run the Task 2 cell.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        job_def = job_resp["Job"]

        expected_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
        actual_role = job_def.get("Role", "")
        # Glue returns the role as either the bare name or the full ARN depending
        # on how it was created. Accept either form so a student who passed
        # just ROLE_NAME does not get a false fail.
        if actual_role not in (expected_role_arn, ROLE_NAME):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job Role is {actual_role!r}, expected {expected_role_arn!r} "
                    f"(or the bare role name {ROLE_NAME!r})."
                ),
            )

        command = job_def.get("Command", {})
        if command.get("Name") != "glueetl":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Command.Name is {command.get('Name')!r}, expected 'glueetl'. "
                    f"glueetl is the PySpark ETL job type; pythonshell and gluestreaming "
                    f"are different job types."
                ),
            )

        expected_script = f"s3://{BUCKET_NAME}/scripts/transform.py"
        if command.get("ScriptLocation") != expected_script:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Command.ScriptLocation is {command.get('ScriptLocation')!r}, "
                    f"expected {expected_script!r}."
                ),
            )

        glue_version = job_def.get("GlueVersion", "")
        if glue_version != "4.0":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"GlueVersion is {glue_version!r}, expected '4.0'. Glue 4.0 is what "
                    f"the lab and the cert YAML risks list pin to."
                ),
            )

        worker_type = job_def.get("WorkerType", "")
        if worker_type != "G.1X":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"WorkerType is {worker_type!r}, expected 'G.1X'. G.1X is the "
                    f"smallest-and-cheapest worker for this dataset size."
                ),
            )

        num_workers = job_def.get("NumberOfWorkers", 0)
        lo, hi = REQUIRED_NUM_WORKERS_RANGE
        if not (lo <= num_workers <= hi):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"NumberOfWorkers is {num_workers}, expected between {lo} and {hi} inclusive."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the job definition in order: Role, Command.Name, Command.ScriptLocation, GlueVersion, WorkerType, NumberOfWorkers. Read the failure reason. The most common miss is the role ARN not matching: a student who passed the bare role name to `create_job` should still pass (the validator accepts both forms), but a student who copy-pasted a stale ARN from a prior account will see the mismatch. The next most common miss is `Command.Name` being left as a default; it must be the string `glueetl`.

</details>

<details><summary>Hint 2 (direction)</summary>

Four API calls in this task. `s3.put_object` uploads `PYSPARK_SCRIPT.encode("utf-8")` to the `scripts/transform.py` key. `iam.create_role` takes the prebuilt `trust_policy` (json-dumped) and tags the role. `iam.put_role_policy` attaches the `inline_policy` (also json-dumped) under `INLINE_POLICY_NAME`. `glue.create_job` ties everything together: `Name=JOB_NAME`, `Role=role_arn`, `Command={"Name": "glueetl", "ScriptLocation": script_uri, "PythonVersion": "3"}`, `GlueVersion="4.0"`, `WorkerType="G.1X"`, `NumberOfWorkers=2`, `DefaultArguments={"--BUCKET_NAME": BUCKET_NAME}`. The 15-second IAM propagation sleep is already in the cell between the role attach and the job create.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2 (boto3 calls only; the script body, trust policy, and inline policy are already defined as constants in the cell):

```python
s3.put_object(
    Bucket=BUCKET_NAME,
    Key=SCRIPT_KEY,
    Body=PYSPARK_SCRIPT.encode("utf-8"),
)

iam.create_role(
    RoleName=ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)

glue.create_job(
    Name=JOB_NAME,
    Role=role_arn,
    Command={
        "Name": "glueetl",
        "ScriptLocation": script_uri,
        "PythonVersion": "3",
    },
    DefaultArguments={
        "--BUCKET_NAME": BUCKET_NAME,
        "--enable-metrics": "true",
    },
    GlueVersion="4.0",
    WorkerType="G.1X",
    NumberOfWorkers=2,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
```

If `glue.create_job` returns `InvalidInputException: User: ... is not authorized to perform: iam:PassRole`, your labrun-test IAM user needs the `iam:PassRole` action on the job role. `IAMFullAccess` already grants it; if you opted into a narrower attach set, add a tiny inline policy granting `iam:PassRole` on the job role ARN. If the call returns `InvalidInputException: The role ARN is not valid` and the role exists, the 15-second propagation wait was not enough; sleep another 15 seconds and retry.

</details>


**Wallet check.** Still essentially $0.00. Creating an IAM role and a Glue database is free. Uploading a 3 KB PySpark script to S3 is free under the always-free request budget. The Glue ETL job is created but has not run yet; no DPU-hours are accruing. Coffee continues to win.


## Task 3: Start the job, wait for it to finish, and confirm success

Now run the job. One API call kicks it off:

```python
run = glue.start_job_run(JobName=JOB_NAME)
run_id = run["JobRunId"]
```

The poll loop in the cell below queries `glue.get_job_run` until the run hits a terminal state (`SUCCEEDED`, `FAILED`, `STOPPED`, or `TIMEOUT`). Glue ETL has a 10-minute minimum billable time per run, so this lab cell stays open for up to 12 minutes of wall-clock. The actual transform on 360 rows finishes in well under a minute; the rest is Glue's worker provisioning and the billing minimum.

If the run fails, the cell prints the CloudWatch log group name and the `ErrorMessage` from `get_job_run`. The most common first-run failure is one of the three date-format parsers in the script not matching the seeded data (PySpark returns NULL on a bad parse, which collapses to NULL `created_at_ts`, which collapses to NULL `year` and `month` partition columns, which the partition write rejects). The PySpark code in Task 2's `PYSPARK_SCRIPT` covers all three formats; if you customized the script, double-check the `coalesce` of three `to_timestamp` calls.

Checkpoint 3 inspects the most recent run and asserts `JobRunState == "SUCCEEDED"` plus `ExecutionTime > 0`. A run still in `RUNNING` or `STARTING` returns a `fail` with a directional nudge to wait, not a passive timeout.


In [ ]:
# NBVAL_SKIP
# Task 3: Start the Glue ETL job and poll until it reaches a terminal state.

glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

TERMINAL_STATES = {"SUCCEEDED", "FAILED", "STOPPED", "TIMEOUT"}

# YOUR CODE: Start the job with run = glue.start_job_run(JobName=JOB_NAME)
# and capture run_id = run["JobRunId"].

print(f"Job run submitted: {JOB_NAME} (run id will appear below)")
print("Asking Glue to spin up workers; this takes a minute or so...")

deadline = time.time() + 720  # 12 minutes
last_state = None
run_resp = None
while time.time() < deadline:
    run_resp = glue.get_job_run(JobName=JOB_NAME, RunId=run_id)
    state = run_resp["JobRun"]["JobRunState"]
    if state != last_state:
        print(f"  state: {state}")
        last_state = state
    if state in TERMINAL_STATES:
        break
    time.sleep(15)
else:
    raise RuntimeError(
        f"Job run {run_id} did not reach a terminal state within 12 minutes. "
        f"Inspect the run in the AWS Glue console."
    )

final_state = run_resp["JobRun"]["JobRunState"]
if final_state == "SUCCEEDED":
    exec_seconds = run_resp["JobRun"].get("ExecutionTime", 0)
    print(f"Run finished SUCCEEDED in {exec_seconds} seconds of execution time.")
else:
    err = run_resp["JobRun"].get("ErrorMessage", "(no error message)")
    log_group = run_resp["JobRun"].get("LogGroupName", "/aws-glue/jobs")
    print(f"Run finished {final_state}: {err}")
    print(f"CloudWatch logs: {log_group}/output and {log_group}/error")
    print("Fix the script, re-upload it via s3.put_object, and re-run this cell.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: most-recent job run for the lab's Glue job finished
# SUCCEEDED and has a non-zero ExecutionTime. RUNNING or STARTING returns
# fail with a clear nudge to wait rather than a passive timeout.

TERMINAL_STATES = {"SUCCEEDED", "FAILED", "STOPPED", "TIMEOUT"}


def checkpoint_3(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            runs = glue_client.get_job_runs(JobName=JOB_NAME, MaxResults=1)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Glue job {JOB_NAME} does not exist. Run the Task 2 cell.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        job_runs = runs.get("JobRuns", [])
        if not job_runs:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job {JOB_NAME} has no runs yet. Call glue.start_job_run "
                    f"in the Task 3 cell first."
                ),
            )

        latest = job_runs[0]
        state = latest.get("JobRunState", "UNKNOWN")
        if state not in TERMINAL_STATES:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Most-recent run is in state {state!r}. Wait until the run "
                    f"finishes (SUCCEEDED, FAILED, STOPPED, or TIMEOUT) before "
                    f"re-running this checkpoint. The Task 3 cell polls until "
                    f"terminal; if you advanced early, run it again."
                ),
            )

        if state != "SUCCEEDED":
            err = latest.get("ErrorMessage", "(no error message)")
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Most-recent run finished {state}: {err}. Inspect the run "
                    f"in the Glue console or CloudWatch Logs and rerun after fixing."
                ),
            )

        if latest.get("Errored") is True:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Most-recent run is marked SUCCEEDED but Errored=True. "
                    "Inspect the run in the Glue console."
                ),
            )

        exec_time = latest.get("ExecutionTime", 0)
        if exec_time <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Most-recent run has ExecutionTime={exec_time}, expected > 0. "
                    f"The Glue API has not finalized the metric yet; wait 30 seconds "
                    f"and re-run this checkpoint."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads the most recent run via `get_job_runs(MaxResults=1)`, then peeks at `JobRunState`, `Errored`, and `ExecutionTime`. If you see a `fail` saying the run is still in `RUNNING` or `STARTING`, the Task 3 cell broke out of its poll early (or the cell ran out of time at 12 minutes). Either re-run the Task 3 cell or wait another minute and re-run this checkpoint. If you see `FAILED`, follow the CloudWatch logs link in the Task 3 cell output.

</details>

<details><summary>Hint 2 (direction)</summary>

One line for the student in this task: `run = glue.start_job_run(JobName=JOB_NAME)` and `run_id = run["JobRunId"]`. The poll loop is already written. The script reads `--BUCKET_NAME` from the job's `DefaultArguments`, so no extra arguments need to be passed to `start_job_run`. If you want to pass `Arguments={...}` you can, but the defaults set in Task 2 cover this lab.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3 (the poll loop below the call is unchanged):

```python
run = glue.start_job_run(JobName=JOB_NAME)
run_id = run["JobRunId"]
```

If the run lands in `FAILED` with `AnalysisException: cannot resolve 'created_at'`, the PySpark `option("header", "true")` is missing from the `spark.read.csv` call; the script in Task 2 sets it explicitly. If the run lands in `FAILED` with `Job aborted due to stage failure ... cannot cast string to double`, the epoch-seconds `to_timestamp` is being applied to non-numeric values; verify the `coalesce` order matches the cell. If it lands in `FAILED` with `s3a:// ... Access Denied`, the inline policy on the role is missing `s3:GetObject` or `s3:PutObject` on `bucket/*`; recheck Task 2 Hint 3.

</details>


**Wallet check.** This is the only cell that materially moves the bill. Glue ETL at 2 G.1X workers bills at $0.44 per DPU-hour with a 10-minute minimum; that comes out to about $0.15 per run. If you re-ran the job once after a script tweak, you are at about $0.30. The cleanup cell at the bottom of this notebook deletes the job so no further DPU-hours accrue.


## Task 4: Register the curated table and prove the Parquet output is partitioned correctly

The job wrote Parquet under `s3://{bucket}/curated/year=YYYY/month=MM/`. Two things to do before the lab is done:

1. **Register the curated table** in the Glue Data Catalog under the database `labrun_glue_etl_transforms_db` so downstream labs (especially Lab 05 with Athena) can query it without re-crawling. The columns are `order_id`, `customer_id`, `amount_cents`, `currency`, `processor_name`, `created_at`. The partition columns are `year` (int) and `month` (int).
2. **Confirm the partition layout is correct.** The checkpoint walks every object under `curated/`, asserts the key matches `year=\d{4}/month=\d{2}/.*\.parquet`, counts distinct `(year, month)` partition values, and (lazily importing `pyarrow`) reads one Parquet file to verify `created_at` is an ISO 8601 string and the `UNKNOWN` sentinel appears in the `customer_id` column for at least one row.

The total deduplicated row count target comes from the raw CSVs: each raw row corresponds to a unique `order_id` after dedup, so the validator reads the three raw CSVs, counts distinct `order_id` values, and asserts the Parquet row count matches within 1 row of tolerance.


In [ ]:
# NBVAL_SKIP
# Task 4: Register the curated Parquet table in the Glue Data Catalog.
# The PySpark job already wrote the Parquet objects under curated/.

glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

table_input = {
    "Name": CURATED_TABLE_NAME,
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {"classification": "parquet", "has_encrypted_data": "false"},
    "StorageDescriptor": {
        "Columns": [
            {"Name": "order_id", "Type": "string"},
            {"Name": "customer_id", "Type": "string"},
            {"Name": "amount_cents", "Type": "int"},
            {"Name": "currency", "Type": "string"},
            {"Name": "processor_name", "Type": "string"},
            {"Name": "created_at", "Type": "string"},
        ],
        "Location": f"s3://{BUCKET_NAME}/curated/",
        "InputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe",
        },
    },
    "PartitionKeys": [
        {"Name": "year", "Type": "int"},
        {"Name": "month", "Type": "int"},
    ],
}

# YOUR CODE: Register the curated table with glue.create_table(
#   DatabaseName=DATABASE_NAME,
#   TableInput=table_input,
# ). If the table already exists from a prior run inside the same session,
# wrap the call in try/except ClientError and reuse on AlreadyExistsException.

print(f"Curated table registered: {DATABASE_NAME}.{CURATED_TABLE_NAME}")
print(f"  Location: s3://{BUCKET_NAME}/curated/")
print(f"  Partition keys: year (int), month (int)")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Parquet output is partitioned by year and month, has at
# least 2 distinct (year, month) partition values, the dedup row count
# matches raw distinct order_ids, and one Parquet file shows ISO 8601
# created_at + the UNKNOWN sentinel for customer_id.

PARQUET_KEY_RE = re.compile(
    r"^curated/year=(\d{4})/month=(\d{1,2})/[^/]+\.parquet$"
)
ISO_TS_RE = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")


def checkpoint_4(session):
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # 1. List Parquet objects under curated/ and check the key pattern.
        paginator = s3_client.get_paginator("list_objects_v2")
        partition_set: set[tuple[str, str]] = set()
        parquet_keys: list[str] = []
        mismatched: list[str] = []
        for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="curated/"):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                # Skip Spark _SUCCESS markers and dot files.
                base = key.rsplit("/", 1)[-1]
                if base.startswith("_") or base.startswith("."):
                    continue
                m = PARQUET_KEY_RE.match(key)
                if not m:
                    mismatched.append(key)
                    continue
                partition_set.add((m.group(1), m.group(2)))
                parquet_keys.append(key)

        if not parquet_keys:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No Parquet objects found under curated/. Confirm the Glue "
                    "job run finished SUCCEEDED and the script writes to "
                    "s3://{bucket}/curated/ with partitionBy('year', 'month')."
                ),
            )
        if mismatched:
            sample = mismatched[:3]
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{len(mismatched)} object key(s) under curated/ do not match "
                    f"year=YYYY/month=MM/<file>.parquet. Sample: {sample}. The most "
                    f"likely cause is the script omitted partitionBy('year', 'month') "
                    f"or used different column names for the partition keys."
                ),
            )
        if len(partition_set) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Only {len(partition_set)} distinct (year, month) partition found. "
                    f"The seeded date range spans multiple months on purpose; check "
                    f"that the script derives year and month from the parsed timestamp "
                    f"and not from a constant."
                ),
            )

        # 2. Row count matches raw distinct order_ids.
        raw_order_ids: set[str] = set()
        for raw_key in ("raw/processor_alpha.csv", "raw/processor_beta.csv", "raw/processor_gamma.csv"):
            obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=raw_key)
            reader = csv.DictReader(io.StringIO(obj["Body"].read().decode("utf-8")))
            for row in reader:
                raw_order_ids.add(row["order_id"])
        expected_rows = len(raw_order_ids)

        # 3. Lazy pyarrow import for the row-count + content check.
        try:
            import pyarrow.parquet as pq  # noqa: F401
            from io import BytesIO
        except ImportError:
            return CheckpointResult(
                status="error",
                error_reason=(
                    "pyarrow is not installed in this Colab runtime. Run "
                    "!pip install pyarrow and re-run this checkpoint."
                ),
            )

        total_rows = 0
        sample_ts: list[str] = []
        seen_unknown_customer = False
        for key in parquet_keys:
            obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
            buf = BytesIO(obj["Body"].read())
            table = pq.read_table(buf)
            total_rows += table.num_rows
            cols = table.column_names
            if "created_at" in cols and table.num_rows > 0:
                for v in table.column("created_at").to_pylist()[:5]:
                    if v is not None:
                        sample_ts.append(str(v))
            if "customer_id" in cols and table.num_rows > 0:
                if "UNKNOWN" in table.column("customer_id").to_pylist():
                    seen_unknown_customer = True

        # Tolerate 1 row of slop in case an edge date row failed all three parsers
        # and got dropped by partition-key NULL handling.
        if abs(total_rows - expected_rows) > 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Parquet has {total_rows} rows but the source has {expected_rows} "
                    f"distinct order_ids. Expected dedup count is the distinct-order_id "
                    f"count from the raw CSVs (one row of slop is allowed for date-parse "
                    f"edge cases). Verify the script calls dropDuplicates(['order_id'])."
                ),
            )

        # 4. ISO 8601 sample.
        if not sample_ts:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No created_at values found in the Parquet sample. The script "
                    "must keep created_at as a column in the output and write it as "
                    "an ISO 8601 string after normalizing the three input formats."
                ),
            )
        non_iso = [v for v in sample_ts if not ISO_TS_RE.match(v)]
        if non_iso:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"created_at values in the Parquet sample are not ISO 8601: "
                    f"{non_iso[:3]}. Expected the form 2024-11-01T00:00:00Z. The "
                    f"script should call date_format on the parsed timestamp."
                ),
            )

        # 5. UNKNOWN sentinel.
        if not seen_unknown_customer:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No row in the Parquet sample has customer_id='UNKNOWN'. The seed "
                    "includes null customer_id rows in every input file and the script "
                    "must fill them with the sentinel string 'UNKNOWN'."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint runs five sub-checks: every curated key matches the partition pattern, at least two distinct `(year, month)` partitions exist, the Parquet row count matches the distinct `order_id` count in the raw CSVs (within one row), every sampled `created_at` is ISO 8601, and at least one Parquet row has `customer_id='UNKNOWN'`. Read the failure reason. If the row count is off, the script's `dropDuplicates(['order_id'])` is missing or keyed on the wrong column. If ISO 8601 fails, the script wrote the raw input format through; `date_format(col('created_at_ts'), ...)` is the fix.

</details>

<details><summary>Hint 2 (direction)</summary>

Only one piece of code in this task: the `glue.create_table` call that registers the curated Parquet table. The `TableInput` dict is already built; pass it with `DatabaseName=DATABASE_NAME` and `TableInput=table_input`. The Parquet itself was written by the Glue job in Task 3; if Checkpoint 4 fails on layout or content, you fix the script (in Task 2's `PYSPARK_SCRIPT`), re-upload it via `s3.put_object`, and re-run the job in Task 3.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
try:
    glue.create_table(
        DatabaseName=DATABASE_NAME,
        TableInput=table_input,
    )
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        pass
    else:
        raise
```

If the underlying Parquet has wrong layout or content, the fix lives in the PySpark script in Task 2. The complete working script body is in Task 2's Hint 3 (or already in the cell as `PYSPARK_SCRIPT`). The five things the script does in order: `spark.read.option('header', 'true').csv(raw_path)`, normalize `created_at` via `coalesce` of three `to_timestamp` calls, reformat `created_at` to ISO 8601 via `date_format`, fill null `customer_id` with `'UNKNOWN'` via `when(...).otherwise(...)`, derive `year` and `month` from the parsed timestamp, `dropDuplicates(['order_id'])`, and write `.partitionBy('year', 'month').parquet(curated_path)`.

</details>


**Wallet check.** Registering a Glue table is a control-plane call and is free. The Parquet reads in the checkpoint validator are S3 GET requests on a handful of small files; well inside the always-free tier. Cleanup is next.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. Lab 04 has no critical (hourly-billed) resources.
#
# labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table. It does NOT yet support glue_job. A
# LabAdapter subclass wraps AwsCleanupAdapter and adds glue_job as an
# inline fallback. When the package promotes glue_job in a future release,
# the wrapper can be removed and run_cleanup called against the default.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    '''AwsCleanupAdapter wrapper that adds glue_job support.'''

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "glue_job":
            client = boto3.client(
                "glue",
                aws_access_key_id=credentials["aws_access_key_id"],
                aws_secret_access_key=credentials["aws_secret_access_key"],
                aws_session_token=credentials.get("aws_session_token"),
                region_name=credentials.get("region", REGION),
            )
            try:
                client.delete_job(JobName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return
                raise
            return
        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "glue_job":
            client = boto3.client(
                "glue",
                aws_access_key_id=credentials["aws_access_key_id"],
                aws_secret_access_key=credentials["aws_secret_access_key"],
                aws_session_token=credentials.get("aws_session_token"),
                region_name=credentials.get("region", REGION),
            )
            try:
                client.get_job(JobName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise
        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. S3 buckets cannot be
# deleted while they contain objects; the bundled s3_bucket adapter only
# deletes one page of objects at a time, and the Glue job wrote curated
# Parquet under multiple partitions plus a _SUCCESS marker plus the raw
# CSVs plus scripts/transform.py. Empty here so the adapter call succeeds.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: $0.15 to $0.50.** Glue ETL job runs are the only line item that materially registers. One successful run at 2 G.1X workers for the 10-minute Glue minimum is about $0.15. Two or three runs while you debug the date parser put you in the $0.30 to $0.50 range. Everything else (S3 storage and requests, IAM, Glue Data Catalog, CloudWatch Logs) is rounding error against the DPU-hours. The cleanup cell above deleted the job, the curated table, the database, the role, and the bucket so the bill stops accruing now.


## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. The PySpark script you shipped reads three input files into one DataFrame using `spark.read.option('header', 'true').csv(raw_path)`. That works at this volume because the files are tiny and the headers happen to be identical. At 100 GB per file and headers that drift over time, Glue's `spark.read.csv` against a wildcard path can be slow and fragile. Walk through the trade-offs: when does it pay to read each file separately with a per-file schema and union them, versus a single multi-source read; when does it pay to crawl the files first and use `glueContext.create_dynamic_frame.from_catalog` instead. This maps to DEA-C01 Domain 1 expectations on Glue PySpark vs. DynamicFrames.

2. The 10-minute Glue billing minimum was the dominant line item on this lab. Identify two situations where you would deliberately structure your pipeline to run one bigger job instead of three smaller ones (to amortize the minimum) and two situations where the right answer is the opposite (smaller, more frequent jobs with bookmarks). Lab 07 covers bookmarks end to end; this question is the warm-up.

3. Your script normalizes three date formats with `coalesce` of three `to_timestamp` calls. The order matters: PySpark evaluates left to right and stops at the first non-null. Walk through what happens if the second `to_timestamp` (the epoch one, cast-to-long) is moved to position 1. Hint: it interacts badly with the ISO 8601 column because of how Spark casts strings to long in ANSI-off mode. This is the kind of subtle ordering bug DEA-C01 likes to test in scenario questions on data quality.

## Exam-Style Practice

**Q1.** A data engineer creates a Glue ETL job that reads three CSV files with inconsistent date formats and writes Parquet partitioned by year and month. The job runs SUCCEEDED, but every output object lands under a single partition `year=__HIVE_DEFAULT_PARTITION__/month=__HIVE_DEFAULT_PARTITION__/`. What is the most likely cause?

A) The Parquet output format does not support partition columns derived from a timestamp; switch to ORC.

B) The script's `coalesce` of `to_timestamp` calls returned NULL for every row, so the derived `year` and `month` columns were NULL and Spark wrote them to the Hive default partition.

C) The Glue job's `WorkerType` is `G.1X` which only supports unpartitioned writes; switch to `G.2X`.

D) The S3 bucket policy denies `s3:PutObject` on partitioned keys; the default partition is a fallback path.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Parquet supports partition columns from any source, including derived timestamp columns. ORC and Parquet have equivalent partitioning semantics in Glue.
- B) Right. When `coalesce(to_timestamp(...), to_timestamp(...), to_timestamp(...))` returns NULL for a row (none of the three formats matched), the `year()` and `month()` functions on that NULL column also return NULL, and Spark routes those rows to `__HIVE_DEFAULT_PARTITION__`. Fix the date-parsing logic.
- C) Wrong. Worker type controls parallelism and memory, not partitioning. G.1X writes partitioned Parquet just fine.
- D) Wrong. An S3 deny would surface as a `Access Denied` job failure, not as the Hive default partition. The default partition is a Hive/Spark convention for NULL partition keys, unrelated to bucket policy.

</details>

**Q2.** A Glue ETL job needs to dedup a DataFrame on `order_id`, keeping the most recently updated row per `order_id` based on an `updated_at` timestamp column. The script currently calls `df.dropDuplicates(['order_id'])` and a code review flags it as incorrect. Which option correctly implements the intended dedup?

A) Replace `dropDuplicates(['order_id'])` with `dropDuplicates(['order_id', 'updated_at'])`.

B) Sort by `updated_at` descending and then call `dropDuplicates(['order_id'])`; PySpark guarantees the first occurrence is retained.

C) Use a window function partitioned by `order_id` ordered by `updated_at` descending, add a row_number, and filter to row_number=1.

D) `dropDuplicates(['order_id'])` is already correct; the reviewer is mistaken.

<details><summary>Show answer</summary>

**C**.

- A) Wrong. `dropDuplicates(['order_id', 'updated_at'])` keeps any row that is unique on the combined `(order_id, updated_at)` key, so two rows with the same `order_id` but different `updated_at` would both survive. That defeats the dedup.
- B) Wrong. PySpark does not guarantee that `dropDuplicates` retains the first row after a sort. The implementation is order-insensitive across partitions, so sorting before dropping is a common but unreliable pattern. The reviewer is right to flag it.
- C) Right. The window-function pattern (`row_number().over(Window.partitionBy('order_id').orderBy(col('updated_at').desc()))` then filter to `row_number=1`) is the idiomatic and reliable way to dedup on a business key while keeping the row with the maximum value of another column.
- D) Wrong. `dropDuplicates` on a single column collapses to keep an arbitrary row per group, not necessarily the most recent. The reviewer is correct.

</details>

**Q3.** A data engineer needs to write the same dataset to S3 in Parquet format twice per day, partitioned by year and month. Each run overwrites the partition for the current month and leaves prior months untouched. The current script uses `df.write.mode('overwrite').partitionBy('year', 'month').parquet(path)` and a stakeholder reports that yesterday's data disappeared after this morning's run. What is the right fix?

A) Switch from `partitionBy('year', 'month')` to `partitionBy('day')` to isolate writes per day.

B) Set Spark config `spark.sql.sources.partitionOverwriteMode='dynamic'` so `overwrite` only touches the partitions present in the new DataFrame.

C) Replace `mode('overwrite')` with `mode('append')`; partition-aware overwriting is not supported in Glue.

D) Write to a versioned S3 prefix per run and stitch the partitions together in Athena via `MSCK REPAIR TABLE`.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Switching partition granularity does not address the overwrite semantics; the same problem would happen at the day boundary.
- B) Right. Spark's default `partitionOverwriteMode` is `static`, which deletes every partition under the table root before writing. Switching to `dynamic` makes `overwrite` only delete and replace the partitions that the new DataFrame writes to, preserving prior partitions. This is the canonical Glue ETL pattern for incremental partition writes.
- C) Wrong. `append` writes new files alongside the old ones without removing them, which creates duplicates within the current month rather than fixing the loss of prior months.
- D) Wrong. A versioned-prefix workaround sidesteps the actual issue and breaks the partitioned-table contract downstream queries depend on. The clean fix is the partition-overwrite-mode config.

</details>
